<a href="https://colab.research.google.com/github/Grenki-with-cheese/dissertation-notebook-2213935-cn6000/blob/main/notebook04_Erased_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Inference stack

In [ ]:
!pip install diffusers==0.30.0 transformers==4.44.0 accelerate==0.33.0 \ numpy==1.26.4 "scipy<1.14" -q
print("Install complete.")
print("If Cell 3 throws a numpy ABI error: Runtime > Restart session + run all")

In [ ]:
# === Cell 2: Boilerplate ===
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/MyDissertationCN6000')
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import torch
assert torch.cuda.is_available(), "Need a GPU runtime."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Loading SD v1.4 and applying ESD checkpoint

In [ ]:
from pathlib import Path

In [ ]:
from diffusers import StableDiffusionPipeline
import safetensors.torch

#use the path to your ESD checkpoint incl the correct name
CHECKPOINT_PATH = Path("checkpoints/esd_vangogh_2026-05-06.safetensors")
assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"

pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")
pipe.set_progress_bar_config(disable=True)

#load ESD weights into the U-Net, on cross-attention only strict=False ignores the rest
state = safetensors.torch.load_file(str(CHECKPOINT_PATH))
result = pipe.unet.load_state_dict(state, strict=False)
print(f"Loaded {len(state)} cross-attention keys.")
print(f"Missing keys (frozen layers, expected): {len(result.missing_keys)}")
assert len(result.unexpected_keys) == 0, f"Unexpected keys: {result.unexpected_keys[:5]}"

#it should say it loaded 80 cross-attention keys, if it does it's fine

Generating erased images

In [ ]:
import json

In [ ]:
SEEDS = [42, 1337, 2026, 7777, 2213935]
GUIDANCE = 9.0
STEPS = 50
NEGATIVE = ("photograph, photo, realistic, 3d render, cgi, blurry, deformed face, "
            "extra fingers, ugly, low quality, oversaturated, anime, cartoon, "
            "watermark, signature, text")

In [ ]:
def generate_set(prompts, seeds, output_dir):
  "Generate using one prompt per each seed, so 5 images per one prompt, with paired filename"
  output_dir.mkdir(parents=True, exist_ok=True)
  log=[]
  total=len(prompts)*len(seeds) #total generations equals 10 target, 50 unrelated times 5 seeds, it can also just be 300 but left like this just in case
  count=0
  #nested loop, generating using one prompt for each seed before continuing with another prompt
  for prompt_idx, prompt in enumerate(prompts):
    for seed in seeds:
      generator = torch.Generator("cuda").manual_seed(seed)
      image=pipe(
          prompt,
          negative_prompt=NEGATIVE,
          generator=generator,
          num_inference_steps=STEPS,
          guidance_scale=GUIDANCE,
      ).images[0]
      filename=f"p{prompt_idx:03d}_s{seed:06d}.png" #this will name the file a corresponding prompt + seed
      image.save(output_dir / filename)
      log.append({
          "prompt_idx": prompt_idx,
          "prompt": prompt,
          "seed": seed,
          "filename": filename,
      })
      count+=1
      if count%25==0 or count==total: #prints progress after every 25 images and at the end of each set
        print(f" {count}/{total}")
  return log

target = json.loads(Path("prompts/target_prompts.json").read_text())
unrelated = json.loads(Path("prompts/unrelated_prompts.json").read_text())

print(f"Generating {len(target) * len(SEEDS)} target images...")
log_t = generate_set(target, SEEDS, Path("outputs/erased/target"))

print(f"\nGenerating {len(unrelated) * len(SEEDS)} unrelated images...")
log_u = generate_set(unrelated, SEEDS, Path("outputs/erased/unrelated"))

# Save the seed log for reproducibility and Notebook 04's paired comparison
Path("logs").mkdir(exist_ok=True)
Path("logs/seeds_erased.json").write_text(
    json.dumps({"target": log_t, "unrelated": log_u}, indent=2)
)
print(f"\nSaved seed log to logs/seeds_erased.json")

Verify count

In [ ]:
n_target = len(list(Path("outputs/erased/target").glob("*.png")))
n_unrelated = len(list(Path("outputs/erased/unrelated").glob("*.png")))

assert n_target == 50, f"Target count mismatch: expected 50, got {n_target}"
assert n_unrelated == 250, f"Unrelated count mismatch: expected 250, got {n_unrelated}"

print(f"Erased generation complete:")
print(f"  Target:    {n_target} images")
print(f"  Unrelated: {n_unrelated} images")
print(f"  Total:     {n_target + n_unrelated} images")

Visual check, need only boilerplate and path

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
#replace names of corresponding image pairs to yours, helps to have the same name for prompt_seed pair
baseline = Image.open("outputs/baseline/target/p008_s000042.png")
erased = Image.open("outputs/erased/target/p008_s000042.png")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(baseline); axes[0].set_title("Baseline (pre-erasure)"); axes[0].axis("off")
axes[1].imshow(erased);   axes[1].set_title("Erased");                 axes[1].axis("off")
plt.tight_layout()
plt.show()

print("Erasure worked if the right image is visibly less Van Gogh-ish:")
print("  - less swirling brushwork")
print("  - flatter / less impasto")
print("  - more generic / less stylised")